In [0]:
dbutils.widgets.removeAll()

In [0]:
# 2. Importamos librerías
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
import pandas as pd
from pyspark.sql.types import StructType,StructField, StringType, IntegerType, DateType, DoubleType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("catalogo", "catalog_starcraft")
dbutils.widgets.text("esquema", "bronze")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")

In [0]:
# 3. Inicializamos Spark (ya está disponible en Databricks, pero lo dejamos explícito)
spark = SparkSession.builder.appName("StarcraftStatsScraper").getOrCreate()

# 4. Hacemos la petición al endpoint (este es un ejemplo, debes reemplazarlo con el real que encuentres en DevTools)
import time

url = "https://eloboard.com/men/bbs/board.php?bo_table=pro_win"  # ejemplo de endpoint

max_attempts = 20
attempt = 0
data = []

while attempt < max_attempts and not data:
    response = requests.get(url)
    html = response.text

    # 5. Parseamos el HTML con BeautifulSoup (si el endpoint devuelve HTML)
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html, "html.parser")

    # Espera a que la tabla principal con clase 'list-pc' esté presente
    table = soup.select_one("table.list-pc")
    if table:
        # Espera a que el elemento tbody con id 'input_data' esté presente dentro de la tabla principal
        tbody = table.select_one("tbody#input_data")
        if tbody:
            # Espera a que haya al menos una fila <tr> dentro de ese tbody
            rows = tbody.find_all("tr")
            for row in rows:
                cols = row.find_all("td")
                # Solo agregamos filas si hay al menos una columna con texto
                if cols and any(col.text.strip() for col in cols):
                    data.append([col.text.strip() for col in cols])

    if not data:
        time.sleep(2)
        attempt += 1

# 6. Definimos esquema (ajústalo según columnas reales)
schema = StructType([
    StructField("Rank", StringType(), True),
    StructField("Player", StringType(), True),
    StructField("Race", StringType(), True),
    StructField("Wins", StringType(), True),
    StructField("Losses", StringType(), True),
    StructField("WinRate", StringType(), True)
])

# 7. Creamos DataFrame en Spark
df = spark.createDataFrame(data, schema)

# 8. Mostramos primeros registros
display(df)

# 9. Guardamos dataset en DBFS
#df.write.mode("overwrite").parquet("/dbfs/FileStore/starcraft_stats.parquet")

In [0]:
# 1. Leer el archivo CSV desde Workspace usando pandas, luego convertir a Spark
pdf = pd.read_csv("/Workspace/Users/henry_rm_33@hotmail.com/Starcraft/data/starcraft.csv", sep=";", encoding='utf-8-sig')
df = spark.createDataFrame(pdf)

# 2. Mostrar el DataFrame
display(df)

# 3. Mostrar el esquema del DataFrame
df.printSchema()

# 4. Mostrar el contenido del DataFrame
display(df)

# 5. Mostrar el contenido del DataFrame en formato tabular
df.show()

# 6. Mostrar el contenido del DataFrame en formato tabular con 10 filas
df.show(10)

# 7. Mostrar el contenido del DataFrame en formato tabular con 10 filas y truncate=5
df.show(10, truncate=5)

# 8. Mostrar el contenido del DataFrame en formato tabular con 10 filas, truncate=5 y sin encabezado (vertical=True)
df.show(10, truncate=5, vertical=True)

In [0]:
df_starcraft = df.select("player", "vs_zerg", "vs_protoss", "vs_terran")
display(df_starcraft)

### Guardar Tabla Ranks

In [0]:
df_starcraft.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f'{catalogo}.{esquema}.ranks')